# Lección 4: Combinación y estructuración de múltiples datasets

**Aprendiz:** Carlos Eduardo Herrera Sibaja
**Programa:** Análisis y Desarrollo de Software
**Evidencia:** GA1-220501094-01-AA1-EV06

Se unen las cinco tablas en una **tabla maestra** en la que cada fila es un producto dentro de un pedido, con su nombre, pasillo, departamento y los datos del pedido. Con ella se responden preguntas que ninguna tabla responde sola. `merge` une **columnas** emparejando claves y `concat` apila **filas**. Un `merge` mal hecho no da error: multiplica o pierde filas en silencio, por eso cada unión se valida.

**Entradas:** `orders_clean.parquet`, `products_clean.parquet` y `order_products_clean.parquet` (generados por [`lesson_2_data_cleaning.ipynb`](lesson_2_data_cleaning.ipynb)), y `aisles.csv` y `departments.csv` de `data/raw/`, que no necesitaron limpieza (Reto 2, ejercicio 2).
**Salidas:** tabla maestra, ficha de productos, líderes por departamento y resúmenes en `data/processed/`.

El notebook forma parte de los laboratorios de Pandas del proyecto Instacart; el índice y el orden de ejecución están en [`pandas_labs.ipynb`](pandas_labs.ipynb). Ejecútelo completo con **Run All**.

## 4.0 Preparación del entorno y carga de los datos

Se cargan las tablas limpias de la Lección 2 y los catálogos de pasillos y departamentos.

In [1]:
from pathlib import Path

import pandas as pd

RUTA = Path('../data/raw')
SALIDA = Path('../data/processed')
necesarios = ['orders_clean.parquet', 'products_clean.parquet', 'order_products_clean.parquet']
faltan = [archivo for archivo in necesarios if not (SALIDA / archivo).exists()]
if faltan:
    raise FileNotFoundError(f'Ejecute primero lesson_2_data_cleaning.ipynb. Faltan: {faltan}')

orders      = pd.read_parquet(SALIDA / 'orders_clean.parquet')
products    = pd.read_parquet(SALIDA / 'products_clean.parquet')
order_prods = pd.read_parquet(SALIDA / 'order_products_clean.parquet')
aisles      = pd.read_csv(RUTA / 'aisles.csv', sep=';')
departments = pd.read_csv(RUTA / 'departments.csv', sep=';')

dias = ['domingo', 'lunes', 'martes', 'miércoles', 'jueves', 'viernes', 'sábado']

print('pandas', pd.__version__)
for nombre, df in [('orders', orders), ('order_prods', order_prods), ('products', products),
                   ('aisles', aisles), ('departments', departments)]:
    print(f'{nombre:<12} {df.shape}')

pandas 3.1.0
orders       (478952, 6)
order_prods  (4545007, 4)
products     (49694, 4)
aisles       (134, 2)
departments  (21, 2)


## 4.1 Ejemplos de la lección

### Tipos de unión con el parámetro `how`

In [2]:
pedidos = pd.DataFrame({'order_id': [1, 2, 3], 'user_id': [10, 20, 30]})
detalle = pd.DataFrame({'order_id': [1, 1, 2, 4], 'product_id': [100, 200, 100, 300]})

for tipo in ['inner', 'left', 'outer']:
    resultado = pedidos.merge(detalle, on='order_id', how=tipo)
    print(tipo, len(resultado))
    print(resultado, '\n')

inner 3
   order_id  user_id  product_id
0         1       10         100
1         1       10         200
2         2       20         100 

left 4
   order_id  user_id  product_id
0         1       10       100.0
1         1       10       200.0
2         2       20       100.0
3         3       30         NaN 

outer 5
   order_id  user_id  product_id
0         1     10.0       100.0
1         1     10.0       200.0
2         2     20.0       100.0
3         3     30.0         NaN
4         4      NaN       300.0 



### Validación de la relación con `validate`

In [3]:
catalogo_malo = pd.concat([products, products.head(1)])

try:
    order_prods.merge(catalogo_malo, on='product_id', validate='m:1')
except pd.errors.MergeError as error:
    print('Detectado:', str(error).splitlines()[0])

Detectado: Merge keys are not unique in right dataset; not a many-to-one merge


### Auditoría de la unión con `indicator`

In [4]:
auditoria = orders.merge(order_prods[['order_id']].drop_duplicates(),
                         on='order_id', how='left', indicator=True)
auditoria['_merge'].value_counts()

_merge
both          450046
left_only      28906
right_only         0
Name: count, dtype: int64

### Construcción de la tabla maestra

In [5]:
filas_inicio = len(order_prods)

maestra = order_prods.merge(products, on='product_id', how='left', validate='m:1')
print('+ products   ', maestra.shape)
maestra = maestra.merge(aisles, on='aisle_id', how='left', validate='m:1')
print('+ aisles     ', maestra.shape)
maestra = maestra.merge(departments, on='department_id', how='left', validate='m:1')
print('+ departments', maestra.shape)
maestra = maestra.merge(orders, on='order_id', how='left', validate='m:1')
print('+ orders     ', maestra.shape)

assert len(maestra) == filas_inicio
assert maestra[['product_name', 'aisle', 'department', 'user_id']].notna().all().all()

+ products    (4545007, 7)


+ aisles      (4545007, 8)


+ departments (4545007, 9)


+ orders      (4545007, 14)


### Preguntas sobre la tabla maestra

In [6]:
top = (maestra.groupby(['product_id', 'product_name']).size()
              .nlargest(5).reset_index(name='veces'))
print(top)

por_depto = (maestra.groupby('department')
                    .agg(articulos=('product_id', 'size'),
                         tasa_recompra=('reordered', 'mean'))
                    .sort_values('articulos', ascending=False))
por_depto.head(3)

   product_id            product_name  veces
0       24852                  Banana  66050
1       13176  Bag Of Organic Bananas  53297
2       21137    Organic Strawberries  37039
3       21903    Organic Baby Spinach  33971
4       47209    Organic Hass Avocado  29773


,articulos,tasa_recompra
department,,
produce,1328302,0.651140
dairy eggs,759842,0.670597
snacks,404850,0.574442


### Apilado con `concat`

In [7]:
categorias = pd.concat([
    aisles.rename(columns={'aisle_id': 'id', 'aisle': 'nombre'}).assign(tipo='pasillo'),
    departments.rename(columns={'department_id': 'id', 'department': 'nombre'}).assign(tipo='departamento'),
], ignore_index=True)
print(categorias.shape)

(155, 3)


### Formato ancho y largo con `pivot_table` y `melt`

In [8]:
ancho = orders.pivot_table(index='order_hour_of_day', columns='order_dow',
                           values='order_id', aggfunc='count')
largo = (ancho.reset_index()
              .melt(id_vars='order_hour_of_day', var_name='order_dow', value_name='pedidos'))
print(ancho.shape, largo.shape)

(24, 7) (168, 3)


### Guardado del resultado

In [9]:
maestra.to_parquet(SALIDA / 'instacart_master.parquet', index=False)
por_depto.to_csv(SALIDA / 'department_summary.csv')
largo.to_csv(SALIDA / 'orders_by_hour_weekday_long.csv', index=False)

## 4.2 Laboratorio guiado 4: tabla de análisis por producto

Encargo del equipo de compras: una tabla con una fila por producto (veces pedido, clientes, tasa de recompra y posición media en el carrito) para encontrar el producto más fiel de cada departamento.

### Paso 1: agregar la tabla maestra por producto

In [10]:
por_producto = (maestra
    .groupby('product_id')
    .agg(veces=('order_id', 'size'),
         clientes=('user_id', 'nunique'),
         tasa_recompra=('reordered', 'mean'),
         posicion_media=('add_to_cart_order', 'mean'))
    .reset_index())
print(por_producto.shape)
por_producto.head(3)

(45573, 5)


,product_id,veces,clientes,tasa_recompra,posicion_media
0,1,280,206,0.564286,5.642857
1,2,11,11,0.000000,8.727273
2,3,42,27,0.738095,6.952381


### Paso 2: recuperar nombre y departamento con `merge`

In [11]:
ficha_productos = (por_producto
    .merge(products, on='product_id', how='left', validate='1:1')
    .merge(departments, on='department_id', how='left', validate='m:1')
    [['product_id', 'product_name', 'department', 'veces', 'clientes',
      'tasa_recompra', 'posicion_media']])
print(ficha_productos.shape)
ficha_productos.loc[ficha_productos['veces'].idxmax()]

(45573, 7)


product_id           24852
product_name        Banana
department         produce
veces                66050
clientes             33274
tasa_recompra     0.844254
posicion_media    4.874815
Name: 22808, dtype: object

### Paso 3: filtrar los productos con suficientes datos

Una tasa de recompra calculada con pocos pedidos no es confiable; se conservan los productos pedidos al menos 1 000 veces.

In [12]:
populares = ficha_productos[ficha_productos['veces'] >= 1000].copy()
print(len(populares))

739


### Paso 4: clasificar dentro de cada departamento con `rank`

In [13]:
populares['puesto'] = (populares
    .groupby('department')['tasa_recompra']
    .rank(ascending=False, method='first'))

lideres = (populares[populares['puesto'] == 1]
           .sort_values('tasa_recompra', ascending=False))
print(len(lideres))
lideres[['department', 'product_name', 'veces', 'tasa_recompra']].head(5)

18


,department,product_name,veces,tasa_recompra
41704,dairy eggs,Whole Organic Omega 3 Milk,1286,0.863142
22808,produce,Banana,66050,0.844254
29030,snacks,Extra Fancy Unsalted Mixed Nuts,1436,0.822423
42302,beverages,Zero Calorie Cola,1256,0.818471
3570,bakery,Organic Honey Sweet Whole Wheat Bread,1607,0.799004


**Interpretación:** el producto más pedido no siempre es el más fiel. En lácteos, la leche Whole Organic Omega 3 Milk tiene pocos pedidos (1 286), pero el 86,3 % de las veces es una recompra; la Banana, con 66 050 pedidos, queda segunda. Para el equipo de compras, estos 18 líderes son productos con demanda estable que no deberían faltar en inventario.

### Paso 5: la misma respuesta con `sort_values` y `head`

In [14]:
lideres_2 = (populares
    .sort_values('tasa_recompra', ascending=False)
    .groupby('department')
    .head(1))
print(len(lideres_2), set(lideres_2['product_id']) == set(lideres['product_id']))

18 True


### Paso 6: guardar la ficha de productos

In [15]:
ficha_productos.to_parquet(SALIDA / 'product_profile.parquet', index=False)
lideres.to_csv(SALIDA / 'department_leaders.csv', index=False)

**Conclusiones clave del laboratorio 4**
- Agregar con `groupby` cambia el nivel de la tabla: de artículos a productos, clientes o días.
- Después de agregar, un `merge` con `validate='1:1'` recupera las columnas descriptivas.
- Filtrar por un mínimo de casos evita conclusiones basadas en pocos datos.
- `groupby().rank()` y `sort_values().groupby().head()` resuelven "el mejor de cada grupo".

## 4.3 Reto 4

Todos los ejercicios usan la tabla `maestra`.

**Ejercicio 1.** Entre los pasillos con al menos 10 000 artículos pedidos, ¿cuáles son los tres con mayor tasa de recompra? ¿Qué pasillo tiene la tasa más baja de todos?

In [16]:
por_pasillo = (maestra.groupby('aisle')
                      .agg(articulos=('product_id', 'size'),
                           tasa_recompra=('reordered', 'mean')))
grandes = por_pasillo[por_pasillo['articulos'] >= 10000]

print('Pasillos con al menos 10 000 artículos:', len(grandes), 'de', len(por_pasillo))
print('\nTop 3 por tasa de recompra:')
print(grandes.nlargest(3, 'tasa_recompra').round(3))
print('\nTasa más baja de todos los pasillos:')
print(por_pasillo.nsmallest(1, 'tasa_recompra').round(3))

Pasillos con al menos 10 000 artículos: 77 de 134

Top 3 por tasa de recompra:
                               articulos  tasa_recompra
aisle                                                  
milk                              125522          0.781
water seltzer sparkling water     117133          0.731
fresh fruits                      510101          0.718

Tasa más baja de todos los pasillos:
                   articulos  tasa_recompra
aisle                                      
spices seasonings      29454          0.152


**Interpretación:** 77 de los 134 pasillos tienen al menos 10 000 artículos. Los de mayor recompra son `milk` (0,781), `water seltzer sparkling water` (0,731) y `fresh fruits` (0,718): productos básicos que se reponen en cada compra. La tasa más baja de todos es la de `spices seasonings` (0,152): las especias duran mucho y se compran pocas veces.

**Ejercicio 2.** Comparar la participación de cada departamento en los pedidos de madrugada (0 a 5 h) con su participación total. ¿Qué departamento entra en el top 3 de madrugada pero no en el total?

In [17]:
madrugada = maestra[maestra['order_hour_of_day'].between(0, 5)]

participacion = pd.DataFrame({
    'madrugada_pct': madrugada['department'].value_counts(normalize=True),
    'total_pct': maestra['department'].value_counts(normalize=True),
}).mul(100).round(2)
participacion['diferencia'] = (participacion['madrugada_pct'] - participacion['total_pct']).round(2)

top3_madrugada = set(participacion['madrugada_pct'].nlargest(3).index)
top3_total = set(participacion['total_pct'].nlargest(3).index)
print('Top 3 madrugada:', sorted(top3_madrugada))
print('Top 3 total:    ', sorted(top3_total))
print('En el top 3 de madrugada pero no en el total:', sorted(top3_madrugada - top3_total))
participacion.sort_values('madrugada_pct', ascending=False).head(6)

Top 3 madrugada: ['beverages', 'dairy eggs', 'produce']
Top 3 total:     ['dairy eggs', 'produce', 'snacks']
En el top 3 de madrugada pero no en el total: ['beverages']


,madrugada_pct,total_pct,diferencia
department,,,
produce,30.21,29.23,0.98
dairy eggs,15.87,16.72,-0.85
beverages,8.49,8.26,0.23
snacks,7.89,8.91,-1.02
frozen,7.65,6.91,0.74
pantry,6.17,5.78,0.39


**Interpretación:** `beverages` entra en el top 3 de madrugada (8,49 %) pero no en el total, donde lo supera `snacks`. En la madrugada ganan peso `produce` (+0,98 puntos) y `frozen` (+0,74), y lo pierden `snacks` (−1,02) y `dairy eggs` (−0,85). Las diferencias son de alrededor de un punto porcentual, así que la mezcla de compras de madrugada es muy parecida a la del resto del día.

**Ejercicio 3.** ¿Cuáles son los tres productos más pedidos del departamento `snacks`?

In [18]:
(maestra[maestra['department'] == 'snacks']
    .groupby(['product_id', 'product_name']).size()
    .nlargest(3)
    .reset_index(name='veces'))

,product_id,product_name,veces
0,13870,Lightly Salted Baked Snap Pea Crisps,3462
1,34448,Original Veggie Straws,2331
2,31040,Sea Salt Pita Chips,2275


**Interpretación:** los snacks más pedidos son Lightly Salted Baked Snap Pea Crisps (3 462), Original Veggie Straws (2 331) y Sea Salt Pita Chips (2 275). Los tres son snacks horneados o de vegetales, en línea con la preferencia por productos orgánicos y saludables que se ve en el top general.

**Ejercicio 4.** Tabla con una fila por cliente y tres columnas: pedidos distintos, artículos y tasa de recompra. ¿Cuántos clientes tiene, cuál es el promedio de artículos por cliente y qué cliente compró más artículos?

In [19]:
por_cliente = (maestra.groupby('user_id')
                      .agg(pedidos=('order_id', 'nunique'),
                           articulos=('product_id', 'size'),
                           tasa_recompra=('reordered', 'mean')))
print('Clientes:', len(por_cliente))
print(f"Promedio de artículos por cliente: {por_cliente['articulos'].mean():.1f}")
print('Cliente con más artículos:', por_cliente['articulos'].idxmax(),
      'con', por_cliente['articulos'].max(), 'artículos')
por_cliente.loc[[por_cliente['articulos'].idxmax()]]

Clientes: 149626
Promedio de artículos por cliente: 30.4
Cliente con más artículos: 103634 con 580 artículos


,pedidos,articulos,tasa_recompra
user_id,,,
103634,18,580,0.822414


**Interpretación:** la tabla tiene 149 626 clientes (los que tienen al menos un pedido con productos) y cada uno compró en promedio 30,4 artículos. El cliente 103634 es el que más compró: 580 artículos en 18 pedidos, con una tasa de recompra de 0,82, es decir, un comprador muy fiel a sus productos.

**Ejercicio 5.** Unir `orders` con los pedidos de `order_prods` usando `how='inner'`. ¿Cuántas filas se obtienen y por qué son menos que con `how='left'`?

In [20]:
pedidos_con_productos = order_prods[['order_id']].drop_duplicates()

union_inner = orders.merge(pedidos_con_productos, on='order_id', how='inner', validate='1:1')
union_left = orders.merge(pedidos_con_productos, on='order_id', how='left',
                          validate='1:1', indicator=True)

print('inner:', len(union_inner))
print('left: ', len(union_left))
print('Diferencia:', len(union_left) - len(union_inner))
print(union_left['_merge'].value_counts())

inner: 450046
left:  478952
Diferencia: 28906
_merge
both          450046
left_only      28906
right_only         0
Name: count, dtype: int64


**Interpretación:** con `inner` se obtienen 450 046 filas y con `left` 478 952: la diferencia son los 28 906 pedidos (6 %) que no tienen productos registrados en `order_prods`. `inner` solo conserva las claves presentes en ambas tablas y los descarta sin aviso; `left` los mantiene y `indicator=True` los muestra como `left_only`.

**Ejercicio 6.** Con `concat`, una sola tabla con los tres productos más pedidos de `produce` y los tres de `dairy eggs`, con una columna que indique el departamento.

In [21]:
def top_productos(departamento, n=3):
    return (maestra[maestra['department'] == departamento]
            .groupby(['product_id', 'product_name']).size()
            .nlargest(n)
            .reset_index(name='veces')
            .assign(department=departamento))

top_produce_dairy = pd.concat([top_productos('produce'), top_productos('dairy eggs')],
                              ignore_index=True)
top_produce_dairy

,product_id,product_name,veces,department
0,24852,Banana,66050,produce
1,13176,Bag Of Organic Bananas,53297,produce
2,21137,Organic Strawberries,37039,produce
3,27845,Organic Whole Milk,19600,dairy eggs
4,49235,Organic Half & Half,10752,dairy eggs
5,27086,Half & Half,9588,dairy eggs


**Interpretación:** en `produce` dominan las bananas y las fresas, con volúmenes muy superiores a los de lácteos: el tercer producto de `produce` (37 039) casi duplica al primero de `dairy eggs` (Organic Whole Milk, 19 600). En lácteos, Half & Half aparece dos veces, en versión orgánica y convencional, porque son productos distintos con su propio `product_id`.

**Ejercicio 7.** Tabla dinámica de artículos por departamento (filas) y día de la semana (columnas). ¿Qué forma tiene y qué día se piden más artículos de `alcohol`?

In [22]:
articulos_depto_dia = maestra.pivot_table(index='department', columns='order_dow',
                                          values='product_id', aggfunc='count')
print('Forma:', articulos_depto_dia.shape)
alcohol = articulos_depto_dia.loc['alcohol']
print('Artículos de alcohol por día:', alcohol.to_dict())
print('Día con más artículos de alcohol:', alcohol.idxmax(), f'({dias[alcohol.idxmax()]})')

articulos_depto_dia.to_csv(SALIDA / 'items_by_department_weekday.csv')

Forma: (21, 7)
Artículos de alcohol por día: {0: 2397, 1: 3025, 2: 2868, 3: 3094, 4: 3656, 5: 3814, 6: 2787}
Día con más artículos de alcohol: 5 (viernes)


**Interpretación:** la tabla tiene forma (21, 7): 21 departamentos por 7 días. El alcohol se compra más el viernes (día 5, 3 814 artículos) y el jueves (3 656), y menos el domingo (2 397). Es el único departamento con el pico fuera del domingo o el lunes (los otros 20 tienen su máximo en uno de esos dos días): el alcohol se compra antes del fin de semana.

---
# Archivos generados

Resumen de los archivos guardados en `data/processed/`:

In [23]:
for archivo in sorted(SALIDA.glob('*')):
    print(f'{archivo.name:<36} {archivo.stat().st_size / 1e6:7.2f} MB')

.gitkeep                                0.00 MB
cleaning_report.csv                     0.00 MB
customer_segments.csv                   0.00 MB
department_leaders.csv                  0.00 MB
department_summary.csv                  0.00 MB
instacart_master.parquet              103.62 MB
items_by_department_weekday.csv         0.00 MB
order_products_clean.parquet           32.51 MB
orders_by_hour_weekday_long.csv         0.00 MB
orders_by_weekday.csv                   0.00 MB
orders_clean.parquet                    5.39 MB
product_profile.parquet                 1.69 MB
products_clean.parquet                  1.45 MB
segment_time_profile.csv                0.00 MB


---
Volver al índice: [`pandas_labs.ipynb`](pandas_labs.ipynb).